# 2024 Ranking Walkthrough

This notebook re-runs the same steps as `scripts/record_vs_diff.py` and `scripts/record_vs_diff_charts.py`, one at a time, so you can watch the data change shape. **It does not write any files** &mdash; every table stays in memory and every chart shows up right here.

The question it answers: *does a 2024 win–loss record line up with how the team actually scored?* Sometimes it does not, and that gap is the whole point.

For the schema and formulas see [feature engineering](../docs/feature-engineering.md); for the findings see [the exploratory data analysis](../docs/exploratory-data-analysis.md).

## Setup and paths

Import the stack (pandas for tables, matplotlib + seaborn for charts) and locate the games CSV. `root` is picked so the notebook runs whether the kernel starts in the repo root or in `notebooks/`. `MISMATCH_RANK_GAP = 6` is the flag threshold used everywhere downstream.

In [ ]:
%matplotlib inline
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
root = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent
GAMES_PATH = root / "data" / "raw" / "nflverse_2024_reg_games.csv"
MISMATCH_RANK_GAP = 6
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 40)
print(f"games CSV: {GAMES_PATH.relative_to(root)}")
print(f"exists: {GAMES_PATH.is_file()}")


## Load the games file

One row per completed 2024 regular-season game. Expect **272 games** across weeks 1&ndash;18 and **32 teams**. Each row is one *game* and names two teams (`home_team` and `away_team`), with the score for each side.

In [ ]:
games = pd.read_csv(GAMES_PATH)
print(f"rows (games): {len(games)}")
print(f"weeks: {int(games['week'].min())}–{int(games['week'].max())}")
print(f"unique home+away teams: {pd.concat([games['home_team'], games['away_team']]).nunique()}")
display(games.loc[[0], ["week", "home_team", "home_score", "away_team", "away_score"]])


## One row per game becomes two rows, one per team: 272 → 544

A game row names two teams, so there is no `team` column to group on yet. `games_to_team_rows` copies each game into **two** rows &mdash; one from the home team's point of view, one from the away team's &mdash; swapping `points_for` and `points_against`. That turns 272 games into 544 team-games and lets us work out `win`, `loss`, and each game's `point_diff`. This 544-row table only ever stays in memory.

In [ ]:
def games_to_team_rows(games: pd.DataFrame) -> pd.DataFrame:
    home_rows = pd.DataFrame(
        {
            "week": games["week"],
            "team": games["home_team"],
            "points_for": games["home_score"],
            "points_against": games["away_score"],
        }
    )
    away_rows = pd.DataFrame(
        {
            "week": games["week"],
            "team": games["away_team"],
            "points_for": games["away_score"],
            "points_against": games["home_score"],
        }
    )
    team_games = pd.concat([home_rows, away_rows], ignore_index=True)
    team_games["win"] = (team_games["points_for"] > team_games["points_against"]).astype(int)
    team_games["loss"] = (team_games["points_for"] < team_games["points_against"]).astype(int)
    team_games["point_diff"] = team_games["points_for"] - team_games["points_against"]
    return team_games
team_games = games_to_team_rows(games)
print(f"rows (team-games): {len(team_games)}")
print(f"columns: {list(team_games.columns)}")
week1_opener = team_games.loc[
    (team_games["week"] == 1) & (team_games["team"].isin(["KC", "BAL"]))
]
display(week1_opener.reset_index(drop=True))


## Summarize: 544 team-games → 32 team-seasons

`groupby("team")` collapses the team-games into one row per team: total `games`, `wins`, `losses`, season `point_diff` (sum of the per-game margins), and `win_pct`. Every team should show `games = 17` (18 calendar weeks minus one bye). DET and KC are shown side by side because they anchor the contrast later &mdash; same record, very different scoring.

In [ ]:
def summarize_season(team_games: pd.DataFrame) -> pd.DataFrame:
    season_totals = (
        team_games.groupby("team", as_index=False)
        .agg(
            games=("win", "size"),
            wins=("win", "sum"),
            losses=("loss", "sum"),
            point_diff=("point_diff", "sum"),
        )
    )
    season_totals["win_pct"] = season_totals["wins"] / season_totals["games"]
    return season_totals
season_totals = summarize_season(team_games)
print(f"rows (teams): {len(season_totals)}")
print(f"games per team: {sorted(season_totals['games'].unique().tolist())}")
display(
    season_totals.loc[season_totals["team"].isin(["DET", "KC"])]
    .sort_values("team")
    .reset_index(drop=True)
)


## Rank twice, then measure the gap

`build_season_table` orders all 32 teams two ways (`rank(method="min")`, so `1` = best and tied teams share a place): once by `win_pct`, once by season `point_diff`. The engineered columns follow:

- `rank_gap` = absolute distance between the two places
- `mismatch` = `rank_gap >= 6` (roughly a fifth of the league)
- `record_ahead_by` = `rank_point_diff - rank_win_pct` (positive means the win ranking sits ahead of the scoring ranking)

In [ ]:
def build_season_table(games: pd.DataFrame) -> pd.DataFrame:
    team_games = games_to_team_rows(games)
    teams = summarize_season(team_games)
    teams["rank_win_pct"] = teams["win_pct"].rank(ascending=False, method="min").astype(int)
    teams["rank_point_diff"] = (
        teams["point_diff"].rank(ascending=False, method="min").astype(int)
    )
    teams["rank_gap"] = (teams["rank_win_pct"] - teams["rank_point_diff"]).abs()
    teams["mismatch"] = teams["rank_gap"] >= MISMATCH_RANK_GAP
    teams["record_ahead_by"] = teams["rank_point_diff"] - teams["rank_win_pct"]
    return teams.sort_values(["rank_win_pct", "team"]).reset_index(drop=True)
teams = build_season_table(games)
print(f"rows: {len(teams)}")
print(f"columns: {list(teams.columns)}")
print(f"mismatch teams: {int(teams['mismatch'].sum())}")
display(teams)


## The headline contrast: KC vs DET

Both finished 15–2, so both rank 1st by record. But KC's `point_diff` was +59 (11th) while DET's was +222 (1st). Same record, a 10-place gap in scoring &mdash; KC trips the mismatch flag, DET does not. This single pair is why neither number, on its own, is team strength.

In [ ]:
pair = teams.loc[
    teams["team"].isin(["DET", "KC"]),
    [
        "team",
        "wins",
        "losses",
        "point_diff",
        "rank_win_pct",
        "rank_point_diff",
        "rank_gap",
        "mismatch",
        "record_ahead_by",
    ],
]
display(pair.reset_index(drop=True))


## The mismatch set

Filter to the flagged teams, largest `record_ahead_by` first. In 2024 there are five (KC, CAR, LA, MIN, HOU) and every one is *record-ahead* &mdash; the win ranking flattered the scoring ranking, never the reverse. That direction is a 2024 fact, not a general law.

In [ ]:
mismatch_teams = teams.loc[teams["mismatch"]].sort_values(
    "record_ahead_by", ascending=False
)
display(mismatch_teams.reset_index(drop=True))


## Charts (Rendered Inline, Not Saved)

Two figures drawn from the 32-team table. The **scatter** plots win percentage against season point differential; gray dots are teams whose two rankings sit close, red dots are the mismatch set (6 or more places apart), and the dashed line is the average point differential at each win percentage. The **bar** chart shows `record_ahead_by` for the five mismatch teams &mdash; how many places each one ranks higher by record than by point differential. The saved script versions of these figures live under `data/figures/` (`record-vs-diff-augmented.png` and `record-ahead-dumbbell.png`); this notebook only displays its own inline versions.

In [ ]:
teams["win_percent"] = teams["win_pct"] * 100
teams["record"] = teams["wins"].astype(int).astype(str) + "–" + teams["losses"].astype(int).astype(str)
teams["label"] = teams["team"] + "  " + teams["record"]
MISMATCH_COLOR = "#c41e3a"
MATCH_COLOR = "#5c6b7a"
TREND_COLOR = "#a8b3bd"
TITLE_COLOR = "#222222"
MUTED_TEXT = "#5a6570"
LABEL_OFFSET = {
    "KC": (16, -26),
    "DET": (12, 12),
    "MIN": (-78, 20),
    "HOU": (16, 20),
    "LA": (16, -36),
    "CAR": (12, 16),
}
def ordinal(n):
    n = int(n)
    if 10 <= (n % 100) <= 20:
        suffix = "th"
    else:
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix}"
def scatter_callout(row):
    record_line = f"{row['team']}  {row['record']}"
    win_line = f"{ordinal(row['rank_win_pct'])} of 32 by record"
    scoring_line = f"{ordinal(row['rank_point_diff'])} of 32 by point differential"
    return f"{record_line}\n{win_line}\n{scoring_line}"
def style_slide_axes(ax: plt.Axes) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#c5ccd3")
    ax.spines["bottom"].set_color("#c5ccd3")
    ax.tick_params(colors=MUTED_TEXT, labelsize=11)
    ax.grid(axis="both", color="#eef1f4", linewidth=1)
    ax.set_axisbelow(True)
sns.set_theme(style="white", context="talk")
fig, ax = plt.subplots(figsize=(11.2, 7.4))
fig.patch.set_facecolor("white")
matching_teams = teams.loc[~teams["mismatch"]]
mismatch_dots = teams.loc[teams["mismatch"]]
sns.regplot(
    data=teams,
    x="win_percent",
    y="point_diff",
    scatter=False,
    ax=ax,
    color=TREND_COLOR,
    line_kws={"linewidth": 2, "linestyle": "--"},
    ci=None,
)
ax.scatter(
    matching_teams["win_percent"],
    matching_teams["point_diff"],
    c=MATCH_COLOR,
    s=64,
    alpha=0.55,
    label="Record and scoring agree",
    zorder=2,
    edgecolors="none",
)
ax.scatter(
    mismatch_dots["win_percent"],
    mismatch_dots["point_diff"],
    c=MISMATCH_COLOR,
    s=130,
    label="Mismatch: 6+ places apart",
    zorder=3,
    edgecolors="white",
    linewidths=0.8,
)
for _, row in teams.loc[teams["team"].isin(LABEL_OFFSET)].iterrows():
    color = MISMATCH_COLOR if row["mismatch"] else MATCH_COLOR
    ax.annotate(
        scatter_callout(row),
        (row["win_percent"], row["point_diff"]),
        textcoords="offset points",
        xytext=LABEL_OFFSET[row["team"]],
        fontsize=9.5,
        color=color,
        fontweight="bold",
        linespacing=1.25,
        bbox={
            "boxstyle": "round,pad=0.28",
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.92,
        },
        arrowprops={"arrowstyle": "-", "color": color, "lw": 0.8, "shrinkA": 1, "shrinkB": 6},
    )
ax.axhline(0, color="#c5ccd3", linewidth=1.2, zorder=1)
ax.axvline(50, color="#c5ccd3", linewidth=1.2, zorder=1)
ax.set_xlabel("Win percentage   →  more wins", fontsize=12, color=TITLE_COLOR)
ax.set_ylabel("Point differential   →  more points scored than allowed", fontsize=12, color=TITLE_COLOR)
fig.suptitle(
    "A 2024 win–loss record does not always match how the team scored",
    fontsize=16,
    fontweight="bold",
    color=TITLE_COLOR,
    x=0.01,
    ha="left",
    y=0.98,
)
ax.set_title(
    "Gray: the two rankings sit close. Red: mismatch (6 or more places apart). "
    "Neither ranking, on its own, is how good a team was.",
    fontsize=11,
    color=MUTED_TEXT,
    loc="left",
    pad=12,
)
ax.legend(frameon=False, loc="lower right", fontsize=10, labelcolor=MUTED_TEXT)
ax.set_xlim(12, 100)
ax.set_ylim(-230, 260)
style_slide_axes(ax)
fig.tight_layout(rect=(0, 0, 1, 0.94))
display(fig)
plt.close(fig)
bar_teams = teams.loc[teams["mismatch"]].sort_values("record_ahead_by", ascending=True).copy()
bar_teams["bar_label"] = (
    bar_teams["team"] + "  " + bar_teams["record"]
    + " · " + bar_teams["rank_win_pct"].map(ordinal) + " of 32 by record · "
    + bar_teams["rank_point_diff"].map(ordinal) + " of 32 by point differential"
)
fig, ax = plt.subplots(figsize=(11.2, 5.8))
fig.patch.set_facecolor("white")
ax.barh(
    bar_teams["bar_label"],
    bar_teams["record_ahead_by"],
    color=MISMATCH_COLOR,
    height=0.62,
    zorder=2,
)
for y, (_, row) in enumerate(bar_teams.iterrows()):
    ax.text(
        row["record_ahead_by"] + 0.12,
        y,
        f"+{int(row['record_ahead_by'])} spots",
        va="center",
        fontsize=11,
        color=TITLE_COLOR,
        fontweight="bold",
    )
ax.set_xlabel(
    "How far record place sat ahead of scoring place",
    fontsize=12,
    color=TITLE_COLOR,
)
ax.set_ylabel("Team", fontsize=12, color=TITLE_COLOR)
fig.suptitle(
    "Record looked better than the scores, 2024",
    fontsize=16,
    fontweight="bold",
    color=TITLE_COLOR,
    x=0.01,
    ha="left",
    y=0.98,
)
ax.set_title(
    "Kansas City’s 15–2 ranked 1st by record and 11th by point differential (+10 ranking spots). "
    "Neither ranking, on its own, is enough.",
    fontsize=11,
    color=MUTED_TEXT,
    loc="left",
    pad=12,
)
ax.set_xlim(0, bar_teams["record_ahead_by"].max() + 2)
style_slide_axes(ax)
ax.grid(axis="x", color="#eef1f4", linewidth=1)
ax.grid(axis="y", visible=False)
fig.tight_layout(rect=(0, 0, 1, 0.93))
display(fig)
plt.close(fig)
